## Example 1: Check the public API exported in __all__

In [ ]:
# Example 1: Check the public API exported in __all__
import rapids_dependency_file_generator as rdfg

print(f"rapids-dependency-file-generator {rdfg.__version__}")
print("__all__:", sorted(rdfg.__all__))

for name in ("parse_config", "load_config_from_file", "make_dependency_files", "Output"):
    assert name in rdfg.__all__, name
assert callable(rdfg.parse_config) and callable(rdfg.load_config_from_file) and callable(rdfg.make_dependency_files)
assert {o.value for o in rdfg.Output} >= {"conda", "requirements", "pyproject"}, list(rdfg.Output)

## Example 2: Configuration file creation and parsing

In [ ]:
import rapids_dependency_file_generator
import tempfile
import yaml
import os

# Example 2: Configuration file creation and parsing
print("Example 2: Dependency configuration management")

try:
    # Create a dependency configuration in the generator's schema: `channels` is a list of names, and
    # each dependency set lists `common` entries of {output_types, packages}.
    dependency_config = {
        'files': {
            'test': {
                'output': 'conda',
                'includes': ['common', 'test_python']
            },
            'build': {
                'output': ['conda', 'requirements'],
                'matrix': {
                    'cuda': ['11.8', '12.0'],
                    'arch': ['x86_64']
                },
                'includes': ['common', 'build_python']
            }
        },
        'channels': ['conda-forge', 'rapidsai', 'nvidia'],
        'dependencies': {
            'common': {'common': [{'output_types': ['conda', 'requirements'], 'packages': ['python>=3.8']}]},
            'test_python': {'common': [{'output_types': ['conda', 'requirements'], 'packages': ['pytest>=6', 'pytest-cov']}]},
            'build_python': {'common': [{'output_types': ['conda', 'requirements'], 'packages': ['setuptools', 'wheel', 'cython']}]}
        }
    }
    
    print(f"Created dependency configuration")
    print(f"Files section: {list(dependency_config['files'].keys())}")
    print(f"Dependencies: {list(dependency_config['dependencies'].keys())}")
    print(f"Channels: {dependency_config['channels']}")
    
    # Create temporary configuration file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as config_file:
        yaml.dump(dependency_config, config_file, default_flow_style=False)
        config_path = config_file.name
    
    print(f"Configuration written to: {os.path.basename(config_path)}")
    
    # Parse it with the installed API: load_config_from_file(path) validates the YAML against the schema
    # and returns a Config; parse_config(config, path) is the dict-level entry point and must agree.
    loaded_config = rapids_dependency_file_generator.load_config_from_file(config_path)
    print(f"Configuration loaded successfully: {type(loaded_config)}")
    assert sorted(loaded_config.files) == ['build', 'test'], loaded_config.files
    assert loaded_config.channels == ['conda-forge', 'rapidsai', 'nvidia'], loaded_config.channels
    assert sorted(loaded_config.dependencies) == ['build_python', 'common', 'test_python'], loaded_config.dependencies
    assert loaded_config.files['build'].matrix == {'cuda': ['11.8', '12.0'], 'arch': ['x86_64']}, loaded_config.files['build'].matrix
    assert rapids_dependency_file_generator.parse_config(dependency_config, config_path) == loaded_config
    
    # Clean up
    os.unlink(config_path)
    print("Temporary configuration file cleaned up")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"rapids_dependency_file_generator example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Generate requirements and conda output with make_dependency_files

In [ ]:
# Example 3: Generate requirements and conda output with make_dependency_files
import contextlib
import io
import os
import tempfile

import yaml
import rapids_dependency_file_generator as rdfg

# One dependencies.yaml shared by Examples 3-5; Example 5 removes the directory.
work = tempfile.TemporaryDirectory()
config_path = os.path.join(work.name, "dependencies.yaml")
with open(config_path, "w") as f:
    yaml.safe_dump({
        "files": {
            "test": {"output": ["conda", "requirements"], "includes": ["common", "test_python"]},
            "build": {
                "output": ["conda", "requirements"],
                "matrix": {"cuda": ["11.8", "12.0"], "arch": ["x86_64"]},
                "includes": ["common", "build_python", "cuda_libs"],
            },
        },
        "channels": ["conda-forge", "rapidsai"],
        "dependencies": {
            "common": {"common": [{"output_types": ["conda", "requirements"], "packages": ["python>=3.8"]}]},
            "test_python": {"common": [{"output_types": ["conda", "requirements"], "packages": ["pytest>=6", "pytest-cov"]}]},
            "build_python": {"common": [{"output_types": ["conda", "requirements"], "packages": ["setuptools", "cython"]}]},
            "cuda_libs": {"specific": [{
                "output_types": ["conda", "requirements"],
                "matrices": [
                    {"matrix": {"cuda": "11.8"}, "packages": ["cupy-cuda11x"]},
                    {"matrix": {"cuda": "12.*"}, "packages": ["cupy-cuda12x"]},
                ],
            }]},
        },
    }, f)

def package_lines(text):
    """The non-comment, non-empty lines of a generated requirements file."""
    return sorted(line for line in text.splitlines() if line and not line.startswith("#"))

cfg = rdfg.load_config_from_file(config_path)

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    rdfg.make_dependency_files(
        parsed_config=cfg, file_keys=["test"], output={rdfg.Output.REQUIREMENTS},
        matrix={"cuda": ["12.0"], "arch": ["x86_64"]}, prepend_channels=[], to_stdout=True,
    )
print(buf.getvalue())
assert package_lines(buf.getvalue()) == ["pytest-cov", "pytest>=6", "python>=3.8"], buf.getvalue()

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    rdfg.make_dependency_files(
        parsed_config=cfg, file_keys=["test"], output={rdfg.Output.CONDA},
        matrix={"cuda": ["12.0"], "arch": ["x86_64"]}, prepend_channels=["my-channel"], to_stdout=True,
    )
env = yaml.safe_load(buf.getvalue())
print(env)
assert env["channels"] == ["my-channel", "conda-forge", "rapidsai"], env
assert sorted(env["dependencies"]) == ["pytest-cov", "pytest>=6", "python>=3.8"], env
assert os.listdir(work.name) == ["dependencies.yaml"]   # to_stdout=True writes no files

## Example 4: Run the command-line entry point

In [ ]:
# Example 4: Run the command-line entry point
# _cli.main is what the rapids-dependency-file-generator command calls; it takes the argument list.
import contextlib
import io

import yaml
from rapids_dependency_file_generator._cli import main

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    main(["--config", config_path, "--file-key", "test", "--output", "requirements",
          "--matrix", "cuda=12.0;arch=x86_64"])
print(buf.getvalue())
assert package_lines(buf.getvalue()) == ["pytest-cov", "pytest>=6", "python>=3.8"], buf.getvalue()

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    main(["--config", config_path, "--file-key", "build", "--output", "conda",
          "--matrix", "cuda=11.8;arch=x86_64", "--prepend-channel", "my-channel"])
env = yaml.safe_load(buf.getvalue())
print(env)
assert env["name"] == "build_cuda-118_arch-x86_64", env["name"]
assert env["channels"][0] == "my-channel"
assert "cupy-cuda11x" in env["dependencies"], env

# --file-key, --output and --matrix must be given together
try:
    main(["--config", config_path, "--file-key", "test", "--output", "requirements"])
except ValueError as e:
    print("Rejected:", str(e).splitlines()[0])
else:
    raise AssertionError("--file-key without --matrix was accepted")

## Example 5: Matrix entries select different packages

In [ ]:
# Example 5: Matrix entries select different packages
import contextlib
import io

import rapids_dependency_file_generator as rdfg

outputs = {}
for cuda in ("11.8", "12.0"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        rdfg.make_dependency_files(
            parsed_config=cfg, file_keys=["build"], output={rdfg.Output.REQUIREMENTS},
            matrix={"cuda": [cuda], "arch": ["x86_64"]}, prepend_channels=[], to_stdout=True,
        )
    outputs[cuda] = package_lines(buf.getvalue())
    print(f"cuda={cuda}: {outputs[cuda]}")

assert outputs["11.8"] == ["cupy-cuda11x", "cython", "python>=3.8", "setuptools"], outputs["11.8"]
assert outputs["12.0"] == ["cupy-cuda12x", "cython", "python>=3.8", "setuptools"], outputs["12.0"]
assert set(outputs["11.8"]) ^ set(outputs["12.0"]) == {"cupy-cuda11x", "cupy-cuda12x"}

work.cleanup()